# RNN
- X_t : [n sample, d dimmension]
- H_t : [n sample, h hidden state]
- W_xh : [d dimmension, h hidden state]
- W_hh : [h, h]


- X_t * W_xh : [n, h]
- H_t-1 * W_hh : [n, h]

- X_t * W_xh + H_t-1 * W_hh : [n, h]

- num_inputs = số lượng feature của mỗi token
- num_hiddens = kích thước bộ nhớ của model, tức là độ rộng của không gian ẩn, vd = 128 nghĩa là dùng vector 128 chiều để lưu


In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [5]:
class RNN(nn.Module):
    def __init__(self, num_samples, num_hidden, sigma=0.01):
        super().__init__()
        self.num_samples = num_samples
        self.num_hiddens = num_hidden
        self.W_xh = nn.Parameter(torch.rand(num_samples, num_hidden) * sigma)
        self.W_hh = nn.Parameter(torch.rand(num_hidden, num_hidden) * sigma)
        self.b_h = nn.Parameter(torch.zeros(num_hidden))

    def forward(self, inputs, state=None):
        if state is None: # H0, từ đầu tiên chưa có hidden state thì khởi tạo = 0
            # Initial state with shape: (batch_size, num_hiddens)
            state = torch.zeros((inputs.shape[1], self.num_hiddens),
                            device=inputs.device)
        else:
            state, = state
        outputs = []
        for X in inputs:  # Shape of inputs: (num_steps, batch_size, num_inputs)
            state = torch.tanh(torch.matmul(X, self.W_xh) +
                            torch.matmul(state, self.W_hh) + self.b_h)
            outputs.append(state)
        return outputs, state

In [6]:
batch_size, num_inputs, num_hiddens, num_steps = 2, 16, 32, 100
rnn = RNN(num_inputs, num_hiddens)
X = torch.ones((num_steps, batch_size, num_inputs))
outputs, state = rnn(X)

In [ ]:
def check_len(a, n):  #@save
    """Check the length of a list."""
    assert len(a) == n, f'list\'s length {len(a)} != expected length {n}'

def check_shape(a, shape):  #@save
    """Check the shape of a tensor."""
    assert a.shape == shape, \
            f'tensor\'s shape {a.shape} != expected shape {shape}'

check_len(outputs, num_steps)
check_shape(outputs[0], (batch_size, num_hiddens))
check_shape(state, (batch_size, num_hiddens))



# Pytorch

In [3]:
import torch
from torch import nn

class RNN(nn.Module):
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()
        self.num_inputs = num_inputs
        self.num_hiddens = num_hiddens
        self.rnn = nn.RNN(input_size=num_inputs,hidden_size=num_hiddens)

    def forward(self, inputs, H=None):
        return self.rnn(inputs, H)

In [4]:
class RNNLanguageModel(nn.Module):
    def __init__(self, num_inputs, num_hiddens, vocab_size):
        super().__init__()
        self.rnn = nn.RNN(input_size=num_inputs, hidden_size=num_hiddens)
        self.linear = nn.Linear(num_hiddens, vocab_size)

    def forward(self, inputs, H=None):
        # inputs: (num_steps, batch_size, num_inputs)
        Y, H = self.rnn(inputs, H)
        # Y: (num_steps, batch_size, num_hiddens)

        # reshape để đưa qua linear
        Y = Y.reshape(-1, Y.shape[-1])  # (num_steps*batch_size, num_hiddens)

        output = self.linear(Y)  # (num_steps*batch_size, vocab_size)

        return output, H